In [23]:
## IMPORTS AND SETUP
# Imports
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
import datetime
import logging
import warnings
import wandb
from PIL import Image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tqdm.notebook import tqdm
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
from wandb.integration.keras import WandbMetricsLogger
from dotenv import load_dotenv

# Force GPU usage
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
print("TensorFlow is using GPU: ", tf.test.is_gpu_available())
print("Devices: ", tf.config.list_physical_devices())
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        pass

# Hide TensorFlow warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0=default, 1=info, 2=warning, 3=error
tf.get_logger().setLevel(logging.ERROR)
tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
tf.debugging.set_log_device_placement(False)
warnings.filterwarnings('ignore')
logging.getLogger('tensorflow').setLevel(logging.FATAL)

# WanDB init
load_dotenv("../.env")
WANDB_API_KEY = os.getenv("API_KEY")
wandb.login(key=WANDB_API_KEY , relogin=True)

I0000 00:00:1744359156.807157     309 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
wandb: WARNING Calling wandb.login() after wandb.init() has no effect.
I0000 00:00:1744359156.807233     309 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744359156.807253     309 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1744359156.807465     309 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-04-11 08:12:36.807502: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2112] Could not identify NUMA node of platform GPU id 0, d

Num GPUs Available:  1
TensorFlow is using GPU:  True
Devices:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]



2025-04-11 08:12:36.807576: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2021] Created device /device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


True

In [24]:
## PARAMETERS
seed = 123
data_format_fix = False # Set to True to fix data formats
data_visualization = False # Set to True to visualize data

# Dataset loading parameters
raw_data_path = "/tf/projet/Dataset" # Path to the raw data folder
excluded_data_folders = ["Dataset Livrable 2"] # Folders to exclude from the dataset
batch_size = 32 # Batch size for dataset loading
img_height = 180 # Image height for dataset loading
img_width = 180 # Image width for dataset loading

# Dataset split parameters
train_split = 0.8 # Proportion of the dataset to use for training
val_split = 0.1 # Proportion of the dataset to use for validation
test_split = 0.1 # Proportion of the dataset to use for testing

# Models loading parameters
models_folder = "/tf/projet/Livrable 1/models" # Path to the models folder
excluded_models = ["CNN.keras"] # Models to exclude from testing

# Models training parameters
learning_rate = 0.001
epochs = 10
history_save_path = models_folder + "/history.json"
save_path = None # Keep None, otherwise will overide existing model

In [25]:
## DATA PREPARATION FUNCTIONS
# Data format fixes
def data_formats_fixes(raw_data_path):
    """
    Walks through a directory to detect and remove problematic image files.
    Removes:
    - Files that are not actually JPG format
    - Corrupted or unreadable images
    Converts:
    - Invalid shape files to RGB format
    Parameters:
    - raw_data_path: Path to the raw data folder.
    """
    print(f"--Starting data format fixes--")

    stats = {
        "processed": 0,
        "wrong_format_removed": 0,
        "invalid_shape_converted": 0, # Includes grayscale
        "corrupted_removed": 0,
        "valid_images": 0
    }

    total_files = 0
    for root, dirs, files in os.walk(raw_data_path):
        for file in files:
            if os.path.splitext(file)[1].lower() == '.jpg':
                total_files += 1

    with tqdm(total=total_files, desc="Checking images") as pbar:
        for root, dirs, files in os.walk(raw_data_path):
            for file in files:
                file_path = os.path.join(root, file)
                _, extension = os.path.splitext(file)

                if extension.lower() != '.jpg':
                    continue

                stats["processed"] += 1
                pbar.update(1)

                try:
                    with open(file_path, 'rb') as f:
                        header = f.read(4)

                    if header[:2] != b'\xff\xd8':  # Not a valid JPEG header
                        os.remove(file_path)
                        stats["wrong_format_removed"] += 1
                        continue

                    try:
                        with Image.open(file_path) as img:
                            if img.mode != 'RGB':
                                img_rgb = img.convert('RGB')
                                img_rgb.save(file_path, 'JPEG', quality=95)
                                stats["invalid_shape_converted"] += 1
                                stats["valid_images"] += 1
                                continue

                            stats["valid_images"] += 1

                    except Exception as e:
                        os.remove(file_path)
                        stats["corrupted_removed"] += 1

                except Exception as e:
                    os.remove(file_path)
                    stats["corrupted_removed"] += 1

    print(f"\nSummary:")
    print(f"Files processed: {stats['processed']}")
    print(f"Wrong format files removed: {stats['wrong_format_removed']}")
    print(f"Invalid shape files converted: {stats['invalid_shape_converted']}")
    print(f"Corrupted files removed: {stats['corrupted_removed']}")
    print(f"Valid images remaining: {stats['valid_images']}")
    print(f"Data format fixes completed.")


# Dataset assembly
def dataset_assembly(raw_data_path, subfolders, batch_size, img_height, img_width, seed):
    """
    Assembles a dataset from a folder structure.
    Parameters:
    - raw_data_path: Path to the raw data folder.
    - subfolders: List of subfolders to include in the dataset.
    Returns:
    - dataset: A TensorFlow dataset object.
    """
    print(f"--Starting dataset assembly--")
    try:
        dataset = tf.keras.utils.image_dataset_from_directory(
            raw_data_path,
            labels="inferred",
            label_mode="int",
            class_names=subfolders,
            color_mode="rgb",
            batch_size=batch_size,
            image_size=(img_height, img_width),
            shuffle=True,
            seed=seed,
            validation_split=None,
            subset=None,
            interpolation="bilinear",
            follow_links=False
        )

        class_names = dataset.class_names
        print(f"Detected classes: {class_names}")

        dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)

        for images, labels in dataset.take(1):
            print(f"Images batch shape : {images.shape}")

        print(f"Dataset assembly completed.")
        return dataset, class_names

    except Exception as e:
        print(f"Error creating dataset: {e}")


# Dataset split
def dataset_split(dataset, train_split=0.8, val_split=0.1, batch_size=1000, seed=123):
    """
    Splits a dataset into training, validation, and test sets.
    Parameters:
    - dataset: The dataset to split.
    - train_split: Proportion of the dataset to use for training.
    - val_split: Proportion of the dataset to use for validation.
    - test_split: Proportion of the dataset to use for testing.
    Returns:
    - train_ds: Training dataset.
    - val_ds: Validation dataset.
    - test_ds: Test dataset.
    """
    print(f"--Starting dataset split--")
    dataset_size = tf.data.experimental.cardinality(dataset).numpy()  # Faster than len(dataset)
    print(f"Dataset size: {dataset_size}")
    train_size = int(train_split * dataset_size)
    val_size = int(val_split * dataset_size)
    test_size = dataset_size - train_size - val_size

    print(f"Creating training set of size ~{train_size*batch_size}...")
    train_ds = dataset.take(train_size)
    remaining_ds = dataset.skip(train_size)
    print(f"Creating validation set of size ~{val_size*batch_size}...")
    val_ds = remaining_ds.take(val_size)
    print(f"Creating test set of size ~{test_size*batch_size}...")
    test_ds = remaining_ds.skip(val_size)

    print(f"Dataset split completed.")
    return train_ds, val_ds, test_ds

In [26]:
## DATA PREPARATION WORKFLOW
# 1. Check dataset existence
print(f"--Checking directory: {raw_data_path}--")
if not os.path.isdir(raw_data_path):
    print(f"Directory {raw_data_path} doesn't exist")
    exit(1)
else:
    print(f"Directory {raw_data_path} exists")

# 2. Fix data formats (if necessary)
if data_format_fix:
    data_formats_fixes(raw_data_path=raw_data_path)
subfolders = [f for f in os.listdir(raw_data_path) if os.path.isdir(os.path.join(raw_data_path, f)) and f not in excluded_data_folders]

# 3. Assemble dataset
dataset, class_names = dataset_assembly(
    raw_data_path=raw_data_path,
    subfolders=subfolders,
    batch_size=batch_size,
    img_height=img_height,
    img_width=img_width,
    seed=seed
)

# 4. Split dataset
train_ds, val_ds, test_ds = dataset_split(
    dataset=dataset,
    train_split=train_split,
    val_split=val_split,
    batch_size=batch_size,
    seed=seed
)

--Checking directory: /tf/projet/Dataset--
Directory /tf/projet/Dataset exists
--Starting dataset assembly--
Found 41376 files belonging to 5 classes.
Detected classes: ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']
Images batch shape : (32, 180, 180, 3)
Dataset assembly completed.
--Starting dataset split--
Dataset size: 1293
Creating training set of size ~33088...
Creating validation set of size ~4128...
Creating test set of size ~4160...
Dataset split completed.


2025-04-11 08:13:00.273598: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [27]:
## DATA VISUALIZATION FUNCTIONS
# Sample visualization
def visualize_class_samples(dataset, class_names, samples_per_class=5):
    """
    Visualize random samples from each class in the dataset.
    Parameters:
    - dataset: TensorFlow dataset
    - class_names: List of class names
    - samples_per_class: Number of samples to display per class
    """
    plt.figure(figsize=(15, 10))

    class_samples = {class_name: [] for class_name in class_names}

    for images, labels in dataset:
        for i, label in enumerate(labels.numpy()):
            class_name = class_names[label]
            if len(class_samples[class_name]) < samples_per_class:
                class_samples[class_name].append(images[i].numpy().astype("uint8"))

        if all(len(samples) >= samples_per_class for samples in class_samples.values()):
            break

    for idx, class_name in enumerate(class_names):
        for i, sample in enumerate(class_samples[class_name]):
            plt.subplot(len(class_names), samples_per_class, idx * samples_per_class + i + 1)
            plt.imshow(sample)
            plt.axis('off')
            if i == 0:
                plt.title(class_name)

    plt.tight_layout()
    plt.show()

# Visualize class distribution
def visualize_class_distribution(dataset, class_names):
    """
    Visualize the distribution of classes in the dataset.
    Parameters:
    - dataset: TensorFlow dataset
    - class_names: List of class names
    """
    class_counts = {class_name: 0 for class_name in class_names}

    for _, labels in dataset:
        for label in labels.numpy():
            class_counts[class_names[label]] += 1

    plt.figure(figsize=(12, 6))
    plt.bar(class_counts.keys(), class_counts.values())
    plt.xlabel('Class')
    plt.ylabel('Count')
    plt.title('Class Distribution')
    plt.xticks(rotation=45)
    plt.show()

In [28]:
## DATA VISUALIZATION WORKFLOW
if data_visualization:
    # 1. Visualize class samples
    visualize_class_samples(dataset=train_ds, class_names=class_names, samples_per_class=5)

    # 2. Visualize class distribution
    visualize_class_distribution(dataset=train_ds, class_names=class_names)

In [29]:
## CALLBACKS PREPARATION
# Confusion matrix callback configuration
class ConfusionMatrixCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_data, class_names):
        super().__init__()
        self.val_data = val_data
        self.class_names = class_names

    def on_epoch_end(self, epoch, logs=None):
        y_true, y_pred = [], []

        # Collect true labels and predictions
        for images, labels in self.val_data:
            preds = self.model.predict(images)
            y_true.extend(labels.numpy())
            y_pred.extend(np.argmax(preds, axis=1))

        # Generate confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        fig, ax = plt.subplots(figsize=(6, 6))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=self.class_names)
        disp.plot(ax=ax, xticks_rotation=45)
        plt.title(f"Confusion Matrix - Epoch {epoch + 1}")
        plt.tight_layout()

        # Log the confusion matrix as an image in wandb
        wandb.log({f"Confusion Matrix (Epoch {epoch + 1})": wandb.Image(fig)}, step=epoch)
        plt.close(fig)

        # Log the raw confusion matrix data for interactive dashboards in wandb
        wandb.log({f"Confusion Matrix Data (Epoch {epoch + 1})": wandb.plot.confusion_matrix(
            probs=None,
            y_true=y_true,
            preds=y_pred,
            class_names=self.class_names
        )}, step=epoch)

# Callbacks initialization
def create_callbacks(model_name="default_model", tensorboard=True, early_stopping=True, model_checkpoint=True, conf_matrix=False, val_data=None, class_names=None):
    """
    TODO: Docstring for this
    """
    log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    callbacks = []

    if tensorboard:
        tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
        callbacks.append(tensorboard_callback)

    if early_stopping:
        early_stopping_callback = tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=4,
            restore_best_weights=True
        )
        callbacks.append(early_stopping_callback)

    if model_checkpoint:
        checkpoint_dir = "checkpoints"
        os.makedirs(checkpoint_dir, exist_ok=True)
        timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(checkpoint_dir, f"{model_name}_{timestamp}.keras"),
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
        callbacks.append(model_checkpoint_callback)

    if conf_matrix and val_data is not None and class_names is not None:
        cm_callback = ConfusionMatrixCallback(val_data=val_data, class_names=class_names)
        callbacks.append(cm_callback)

    return callbacks

In [33]:
## MODELS PREPARATION FUNCTIONS
# Load models
def load_models(models_path, excluded_models=[]):
    """
    Load the .keras models from a folder.
    Parameters:
    - models_path: Path to the folder containing the models
    - excluded_models: List of models to exclude
    Returns:
    - Dictionary with model names as keys and loaded model objects as values
    """
    print(f"--Loading models from {models_path}--")
    models = {}

    # Check if the folder exists
    if not os.path.exists(models_path):
        print(f"Warning: Models folder '{models_path}' does not exist!")
        return models

    # List all .keras files in the directory
    model_files = [f for f in os.listdir(models_path) if f.endswith('.keras') and f not in excluded_models]

    if not model_files:
        print(f"No valid .keras models found in '{models_path}'")
        return models

    # Load each model
    for model_file in model_files:
        model_path = os.path.join(models_path, model_file)
        try:
            print(f"Loading model: {model_file}")
            model = tf.keras.models.load_model(model_path)
            model_name = os.path.splitext(model_file)[0]  # Remove .keras extension
            models[model_name] = model
            print(f"Successfully loaded model: {model_name}")
        except Exception as e:
            print(f"Error loading model {model_file}: {str(e)}")

    print(f"Loaded {len(models)} models: {list(models.keys())}")
    return models


def check_and_compile_model(model, model_name):
    """
    Check if model is compiled and compile it with default settings if not.
    Parameters:
    - model: The model to check
    - model_name: Name of the model (for logging)
    Returns:
    - The (possibly compiled) model
    """
    if not hasattr(model, 'optimizer') or model.optimizer is None:
        print(f"Model {model_name} is not compiled. Compiling with default settings...")
        model.compile(
            optimizer='adam',
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )
    return model


def train_models(models, train_ds, val_ds, epochs=10, save_path=None, history_save_path=None):
    """
    Train multiple models.
    Parameters:
    - models: Dictionary with model names as keys and model objects as values
    - train_ds: Training dataset
    - val_ds: Validation dataset
    - epochs: Number of epochs to train each model
    - save_path: Path to save the trained models (if None, models won't be saved)
    - history_save_path: Path to save the training history (if None, history won't be saved)
    Returns:
    - Dictionary with model names as keys and history objects as values
    """
    histories = {}

    for model_name, model in models.items():
        print(f"\n--Training {model_name}--")

        model = check_and_compile_model(model, model_name)

        history = model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=2
        )

        histories[model_name] = history.history

        if save_path:
            os.makedirs(save_path, exist_ok=True)
            model_save_path = os.path.join(save_path, f"{model_name}.keras")
            model.save(model_save_path)
            print(f"Model saved to {model_save_path}")

    if history_save_path:
        os.makedirs(os.path.dirname(history_save_path), exist_ok=True)
        import pickle
        with open(history_save_path, 'wb') as f:
            pickle.dump(histories, f)
        print(f"Training histories saved to {history_save_path}")

    return histories

In [34]:
## MODEL COMPARISON WORKFLOW
# 1. Load all models
models = load_models(models_path=models_folder, excluded_models=excluded_models)

# 2. Initialize callbacks
run = wandb.init(
    entity="tom-antoine-cesi",
    project="Leyanda",
    name=f"model_comparison_{datetime.datetime.now().strftime('%Y%m%d-%H%M%S')}",
    config={
        "models": list(models.keys()),
        "learning_rate": learning_rate,
        "epochs": epochs,
    }
)
callbacks_dict = {}
for model_name, model in models.items():
    callbacks_dict[model_name] = [WandbMetricsLogger()] + create_callbacks(
        model_name=model_name,
        tensorboard=True,
        early_stopping=True,
        model_checkpoint=True,
        conf_matrix=True,
        val_data=test_ds,
        class_names=class_names
    )

# 2. Train models
train_models(
    models=models,
    train_ds=train_ds,
    val_ds=val_ds,
    epochs=epochs,
    save_path=None,
    history_save_path=history_save_path
)

--Loading models from /tf/projet/Livrable 1/models--
Loading model: CNN_Dropout_3_0.3_BatchNormalization.keras
Successfully loaded model: CNN_Dropout_3_0.3_BatchNormalization
Loading model: CNN_Dropout_4_0.3.keras
Successfully loaded model: CNN_Dropout_4_0.3
Loaded 2 models: ['CNN_Dropout_3_0.3_BatchNormalization', 'CNN_Dropout_4_0.3']



--Training CNN_Dropout_3_0.3_BatchNormalization--
Epoch 1/10


Corrupt JPEG data: 419 extraneous bytes before marker 0xd9
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


1034/1034 - 62s - 60ms/step - accuracy: 0.7136 - loss: 0.6899 - val_accuracy: 0.7970 - val_loss: 0.5438
Epoch 2/10


Corrupt JPEG data: 419 extraneous bytes before marker 0xd9
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


...

Corrupt JPEG data: 419 extraneous bytes before marker 0xd9
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


1034/1034 - 65s - 63ms/step - accuracy: 0.9136 - loss: 0.2149 - val_accuracy: 0.8946 - val_loss: 0.2821
Epoch 10/10


Corrupt JPEG data: 419 extraneous bytes before marker 0xd9
Corrupt JPEG data: 419 extraneous bytes before marker 0xd9


1034/1034 - 65s - 63ms/step - accuracy: 0.9183 - loss: 0.2035 - val_accuracy: 0.9002 - val_loss: 0.2765
Training histories saved to /tf/projet/Livrable 1/models/history.json


{'CNN_Dropout_3_0.3_BatchNormalization': {'accuracy': [0.7135517597198486,
   0.8316912651062012,
   0.8652985692024231,
   0.880953848361969,
   0.8930730223655701,
   0.9012330770492554,
   0.9105415940284729,
   0.9215123057365417,
   0.9279195070266724,
   0.9354146718978882],
  'loss': [0.6898638010025024,
   0.4120784103870392,
   0.3374496400356293,
   0.3008025586605072,
   0.27410873770713806,
   0.2474430501461029,
   0.22830994427204132,
   0.20367206633090973,
   0.18989674746990204,
   0.17193688452243805],
  'val_accuracy': [0.7969961166381836,
   0.8398740291595459,
   0.8689438104629517,
   0.8422965407371521,
   0.8980135917663574,
   0.869428277015686,
   0.8733042478561401,
   0.8994670510292053,
   0.9016472697257996,
   0.9103682041168213],
  'val_loss': [0.5437869429588318,
   0.4084426760673523,
   0.3374188542366028,
   0.40349653363227844,
   0.286754846572876,
   0.3315756320953369,
   0.33649805188179016,
   0.28651610016822815,
   0.28574642539024353,
   0.2